In [2]:
# Import splish
import pandas as pd
import numpy as np
import re
import os
from nltk.inference.prover9 import *

os.environ["PROVER9"] = "/home/flopezp/Prover9/bin/prover9"

In [3]:
folio_full_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
folio_full_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
#trying_splish = pd.read_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/filtered/translation/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv')
#trying_splish = trying_splish.drop(columns = ["Unnamed: 0"])
#trying_splish.head()

In [4]:
def prove(argument):
    goal, assumptions = argument
    g = Expression.fromstring(goal)
    alist = [Expression.fromstring(a) for a in assumptions]
    p = Prover9Command(g, assumptions=alist).prove()
    return p

# FOL to Prover9 Syntax

def switch_quantifiers(text, cuantifier):
    """
        Elimina todos los cuantificadores y los traduce a sintaxis de Prover9. Sin importar la variable ni la cantidad de apariciones. Qué pedo soy una verga para esto.

        text = str ;  Texto a modificar.
        cuantifier = str ('forall', 'exists') ; Cuantificador a modificar.
    """
    if cuantifier == 'forall':
        regex = '∀[A-z]'
        cuant = ' all '
    else:
        regex = '∃[A-z]'
        cuant = ' exists '

    owo = re.finditer(regex, text)
    aux_list = list(owo)
    if len(aux_list) == 0:
        return text

    # Redefinimos el iterador porque hacer lista de un iterador lo consume. CHINGA TU MADRE PYTHON. VETE A LA BURGER.    
    owo = re.finditer(regex, text)
    temporal_str = ''

    for _ in owo:
        if temporal_str == '':
            temporal_str = text[0:_.start()] + cuant + _.group()[-1] + ' ' + text[_.end():]
        else:
            value = re.search(regex, temporal_str)
            temporal_str = temporal_str[0:(value.start())] + cuant + value.group()[-1] + ' ' + temporal_str[value.end():]
    
    return temporal_str



def fol_to_prover9(value):
    """
        Modifica los símbolos lógicos normales y los cambia por los valores adecuados para Prover9.*
    """
    temp = value.lower()
    temp = switch_quantifiers(temp, 'forall')
    temp = switch_quantifiers(temp, 'exists')
    temp = re.sub('-', '', temp)
    temp = re.sub('¬', ' -', temp) 
    temp = re.sub('→|→', '->', temp)
    temp = re.sub('∧', '&', temp)
    temp = re.sub('∨', '|', temp)
    temp = re.sub('↔', '<->', temp)
    temp = re.sub('≠', '!=', temp)
    temp = re.sub(r'\'', '', temp)
    #temp = re.sub(r'[\'\"]', '', temp)
    #temp = re.sub(r'\[', '(', temp)
    #temp = re.sub(r'\]', ')', temp)
    temp = re.sub(r'\[', '', temp)
    temp = re.sub(r'\]', '', temp)
    temp = re.sub('∴', '', temp)
    temp = re.sub(r'(\. \()', '.(', temp)
    temp = re.sub(r'\.{2}', '.', temp)
    #temp = re.sub(r'\)\.', ')', temp)
    temp = re.sub(r'([^a-z]\.\()', '(', temp)
    temp = re.sub(r'\.', ' ', temp)
    temp = re.sub(r'\?', '', temp)
    temp = re.sub(r'[\"\`]+', '', temp)
    #if '/' in temp:
    #    temp = temp[:temp.index('/')]
    #temp = temp + '.'
    return temp


def dash_predicates(text):
    """
        Cambia los predicados de la forma "texto-texto-texto(x)" -> "textotextotexto(x)"

        text = str ; el hilo a modificar.
    """
    all_values = len(re.findall(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', text))

    if all_values == 0:
        return text
    
    new_text = text
    for i in range(all_values):
        current_regex = re.search(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


def elim_spaces(text):
    """
        Elimina los espacios entre variables: lionel messi -> lionelmessi

        text = str; El texto a modificar.

        OBS: Este formato de funciones (Encontrar cantidades y luego iterar sobre las cantidades) me gusta bastante.
    """
    total_iters = len(list(re.finditer(r'([A-z]+ )+([A-z]{2,})', text)))
    if total_iters == 0:
        return text

    new_text = text
    for i in range(total_iters):
        current_regex = re.search(r'([A-z]+ )+([A-z]{2,})', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


# El XOR me tiene hasta los huevos cabrón te lo juro.
def xor_bonito(expression):
    """
        Elimina el símbolo de XOR, y lo reescribe en la fórmula (A OR B) AND NOT(A AND B)
    """
    individual_values = re.findall(r'([A-z|_|0-9]{2,}|¬)', expression)
    a = individual_values[0] + '(x)'
    b = individual_values[1] + '(x)'
    a_or_b = '(' + a + ' | ' + b + ')'
    not_a_and_b = ' -(' + a + ' & ' + b +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def xor_bonito_extreme(expression):
    """
        Elimina los XOR de fórmulas compuestas.
    """
    aux2 = re.search(r'([a-z]+\([a-z, ]+\)) ⊕ ([a-z]+\([a-z, ]+\))', expression)
    split = aux2.group().split('⊕')
    a_f = split[0]
    b_f = split[-1]
    a_or_b = '(' + a_f + ' | ' + b_f + ')'
    not_a_and_b = ' -(' + a_f + ' & ' + b_f +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def rewrite_xor(re_search, element, comp):
    """
        Genera una nueva expresión a partir del xor bonito. 

        re_search = re.search(regex, str)
        element = str ; same str as above
        comp = bool ; True iff predicates have multiple variables.
    """
    start = re_search.start()
    end = re_search.end()

    xor_substr = element[start:end]
    xor_chido = xor_bonito_extreme(xor_substr)
    #if comp:
    #    xor_chido = xor_bonito_extreme(xor_substr)
    #else:
    #    xor_chido = xor_bonito(xor_substr)

    nuevo = element[:start] + xor_chido + element[end:]
    return nuevo


def clean(value, FOLIO):
    """
        Procesa una respuesta individual de FOLIO/GPT_TRANS/QWEN_TRANS para que se pase al formato de Prover9.
    """
    if FOLIO:
        clean_premises_aux = value.split('\n')
    else:
        clean_premises_aux = str(value).split('\', ')
        if len(clean_premises_aux) == 1:
            clean_premises_aux = str(value).split('", ')

    clean_premises = []
    for _ in clean_premises_aux:
        if _ != '':
            clean_premises.append(_)

    if "Premises" in clean_premises[0]:
        del clean_premises[0]

    #print(clean_premises[0])

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub(r'(:::)+([ A-z.\-,⊕\'$0-9\“\”:Śą\’\(\)á//]+)', '', _)

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub(r'[0-9]\.', '', _)
    
    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = fol_to_prover9(instance)

    #print(clean_premises[0])
    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = elim_spaces(instance)

    try:
        # Filtro XOR sencillo
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                element_new = element
                while xor_count > 0:
                    aux = re.search(r'(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element_new)
                    element_new = rewrite_xor(aux, element_new, False)
                    xor_count = len(re.findall('⊕', element_new))
                clean_premises[clean_premises.index(value)] = element_new

    except:
        a = 0    

    for elem in clean_premises:
        aux = elem.split(' , ')
        if len(aux) > 1:
            clean_premises.remove(elem)
            for value in aux:
                clean_premises.append(value)

    if 'folpremises:' in clean_premises:
        while 'folpremises:' in clean_premises:
            clean_premises.remove('folpremises:')

    return clean_premises



# ========================================
# ========================================
# ========================================

def check_arg_validity(dataset, index, validation, path):
    """
        Dadas premisas en lenguaje lógico, verifica que la conclusión correspondiente sea TRANSerible.
    """
    if validation:
        folio_full = folio_full_val
    else:
        folio_full= folio_full_test

    can_parse, neg_parse, prover9_fatal, logical_exp = 0,0,0,0
    
    llm_value = dataset['Translation'][index]
    #folio_value = folio_full['premises-FOL'][index]
    #true_validity = folio_full['label'][index]

    #print("Logical Validity: {}".format(true_validity))

    clean_llm = clean(llm_value, False)
    #clean_folio = clean(folio_value, True)
    #clean_conc = clean(folio_full['conclusion-FOL'][index], True) 
    clean_conc = ['lie(cake)']

    if 'DeepSeek-R1-Distill-Qwen-7B' in path and validation:
        clean_llm = clean_llm[0].split('\n')

    args_llm = (
        clean_conc[0],
        clean_llm
    )

    #args_folio = (
    #    clean_conc[0],
    #    clean_folio
    #)
    
    try:
        prove(args_llm)
        can_parse += 1
    except Exception as e:
        type_error = str(repr(e).split('(')[0])
        print(type_error)
        if type_error == 'Prover9FatalException':
            prover9_fatal+=1
            #print(args_llm)
            print(e)
            #print('-' * 40)
        elif type_error == 'LogicalExpressionException':
            logical_exp+=1
            print(e)
        else:
            owo = 0
            #print('Otro error')
            #print(type_error)
            print(e)
        neg_parse += 1
        
    #try:
    #    print("FOLIO Ejecutable. Valor: {}".format(prove(args_folio)))
    #    fol_valid += 1
    #except Exception as e:
    #    print(e)
    #    print(type(e))
    #    fol_invalid += 1
    #print("======================")

    return can_parse, neg_parse, prover9_fatal, logical_exp
    

In [10]:
baseline_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_DeepSeek-R1-Distill-Qwen-7B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gemma-3-4b-it.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gpt-oss-20b.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-4B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-8B-FP8_NEW.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-14B-FP8_NEW.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3.5-4B_NEW.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3.5-9B_NEW.csv'       
]

#for elem in baseline_path[2]:
dataset = pd.read_csv(baseline_path[1].format('validation'))
if "Retranslation" in dataset.columns:
    dataset = dataset.rename(columns = {"Retranslation": "Translation"})
for i in range(len(dataset['Translation'])):
    print(i)
    aux = str(dataset['Translation'][i]).split('\', ')
    if len(aux) == 1:
        aux = str(dataset['Translation'][i]).split('\", ')
    cleaned = clean(dataset['Translation'][i], None)
    for elem in aux:
        print(elem)
    print(' ')
    cleaned1 = cleaned[0].split('\n')
    for elem in cleaned1:
        print(elem)
    print('='*60)

0
∀x (Club(x) ∧ Performs(x) ∧ SchoolTalentShow(x) → (Attends(x) ∧ Engaged(x) ∧ SchoolEvent(x))) 
∀x (Club(x) ∧ (Performs(x) ∧ Often) ∨ (Inactive(x) ∧ Disinterested(x)))
 
 all x  (club(x) & performs(x) & schooltalentshow(x) -> (attends(x) & engaged(x) & schoolevent(x))) 
 all x  (club(x) & (performs(x) & often) | (inactive(x) & disinterested(x)))
1
∀x (Club(x) ∧ Performs(x) ∧ SchoolTalentShow(x) → (Attends(x) ∧ Engaged(x) ∧ SchoolEvent(x))) 
∀x (Club(x) ∧ (Performs(x) ∧ Often) ∨ (Inactive(x) ∧ Disinterested(x)))
 
 all x  (club(x) & performs(x) & schooltalentshow(x) -> (attends(x) & engaged(x) & schoolevent(x))) 
 all x  (club(x) & (performs(x) & often) | (inactive(x) & disinterested(x)))
2
∀x (Club(x) ∧ Performs(x) ∧ SchoolTalentShow(x) → (Attends(x) ∧ Engaged(x) ∧ SchoolEvent(x)))
∀x (Club(x) ∧ (Performs(x) ∧ Often) ∨ (Inactive(x) ∧ Disinterested(x)))
 
 all x  (club(x) & performs(x) & schooltalentshow(x) -> (attends(x) & engaged(x) & schoolevent(x)))
 all x  (club(x) & (performs(x) 

In [5]:
baseline_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-4B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-8B-FP8_NEW.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-14B-FP8_NEW.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3.5-4B_NEW.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3.5-9B_NEW.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gemma-3-4b-it.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gpt-oss-20b.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_DeepSeek-R1-Distill-Qwen-7B.csv'        
]

alignment_res_path = [
    '/home/flopezp/Kurosagol/Ongoing/alignment_results/{}/filtered/translation/TRANS_KTO_Qwen3-14B.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_results/{}/filtered/translation/TRANS_KTO_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_results/{}/filtered/translation/TRANS_KTO_DeepSeek-R1-0528-Qwen3-8B.csv'
]


def evaluate_parsability(dataset_path, validation):
    if validation:
        path = dataset_path.format('validation')
        split = 'Validation'
        val = True
    else:
        path = dataset_path.format('test')
        val = False
        split = 'Test'

    dataset = pd.read_csv(path)
    if "Unnamed: 0" in dataset.columns:
        dataset = dataset.drop(columns = ["Unnamed: 0"])
    if "Retranslation" in dataset.columns:
        dataset = dataset.rename(columns = {"Retranslation": "Translation"})

    parse_total, nonparse_total, prover9ex_total, logicalex_total = 0, 0, 0, 0
    for i in range(len(dataset['Translation'])):
        parse, nonparse, prover9ex, logicalex  = check_arg_validity(dataset, i, val, path)
        parse_total += parse
        nonparse_total += nonparse
        prover9ex_total += prover9ex
        logicalex_total += logicalex

    model_id = path.split('/')[-1][6:-4]
    length = len(dataset['Translation'])

    print("-"*60)
    print('\t {}'.format(model_id))
    print('\t Split: {}'.format(split))
    print("-"*60)
    print("Valores parseables: {}".format(parse_total))
    print("Valores NO parseables: {}".format(nonparse_total))
    print("Porcentaje parseable: {}".format(round(parse_total/length, 3)))
    print("Arity Errors: {}".format(prover9ex_total))
    print("Syntax Errors: {}".format(logicalex_total))
    print('='*60)

In [8]:
for path in baseline_path:
    evaluate_parsability(path, True)
    evaluate_parsability(path, False)

LogicalExpressionException
Unexpected token: '⊕'.
(attends(x) & engaged(x) & student(x)) ⊕  -(attends(x) & engaged(x) & student(x))
                                       ^
LogicalExpressionException
Unexpected token: '⊕'.
(attends(x) & engaged(x) & student(x)) ⊕  -(attends(x) & engaged(x) & student(x))
                                       ^
LogicalExpressionException
Unexpected token: '⊕'.
(attends(x) & engaged(x) & student(x)) ⊕  -(attends(x) & engaged(x) & student(x))
                                       ^
Prover9FatalException
(FATAL)
%%ERROR: The following symbols are used with multiple arities: monkeypox/1, monkeypox/0.


Fatal error:  The following symbols are used with multiple arities: monkeypox/1, monkeypox/0
Prover9FatalException
(FATAL)
%%ERROR: The following symbols are used with multiple arities: monkeypox/1, monkeypox/0.


Fatal error:  The following symbols are used with multiple arities: monkeypox/1, monkeypox/0
Prover9FatalException
(FATAL)
%%ERROR: The following 

In [9]:
for path in alignment_res_path:
    evaluate_parsability(path, True)
    evaluate_parsability(path, False)

LogicalExpressionException
Unexpected token: '⊕'.  Expected token ')'.
 all x  (performs(x) ⊕ (inactive(x) & disinterested(x)))
                     ^
LogicalExpressionException
Unexpected token: '⊕'.  Expected token ')'.
 all x  (performs(x) ⊕ (inactive(x) & disinterested(x)))
                     ^
LogicalExpressionException
Unexpected token: '⊕'.  Expected token ')'.
 all x  (performs(x) ⊕ (inactive(x) & disinterested(x)))
                     ^
LogicalExpressionException
Unexpected token: '⊕'.
(fromearth(marvin) & frommars(marvin)) ⊕  -(fromearth(marvin) | frommars(marvin))
                                       ^
LogicalExpressionException
Unexpected token: '⊕'.
(fromearth(marvin) & frommars(marvin)) ⊕  -(fromearth(marvin) | frommars(marvin))
                                       ^
LogicalExpressionException
Unexpected token: '⊕'.
(fromearth(marvin) & frommars(marvin)) ⊕  -(fromearth(marvin) | frommars(marvin))
                                       ^
Prover9FatalException
(FATAL

Prover9FatalException
(FATAL)
%%ERROR: Set parsing is not available (see end of marked string):

%%START ERROR%%
    all x (containsonly(x,{%%END ERROR%%s,p,o,t}) -> anagram(x,tops)).

Fatal error:  sread_term error
Prover9FatalException
(FATAL)
%%ERROR: The following symbols are used with multiple arities: tameimpala/1, tameimpala/0.


Fatal error:  The following symbols are used with multiple arities: tameimpala/1, tameimpala/0
Prover9FatalException
(FATAL)
%%ERROR: The following symbols are used with multiple arities: tameimpala/1, tameimpala/0.


Fatal error:  The following symbols are used with multiple arities: tameimpala/1, tameimpala/0
Prover9FatalException
(FATAL)
%%ERROR: The following symbols are used with multiple arities: tameimpala/1, tameimpala/0.


Fatal error:  The following symbols are used with multiple arities: tameimpala/1, tameimpala/0
LogicalExpressionException
Unexpected token: '⊕'.
(expectedcareer(riley) & expectedexam(riley)) ⊕  -(expectedcareer(riley) | expec

In [ ]:
# CFG Generation
from datasets import load_dataset

sergio2_ds = load_dataset('fol-autoformalization/translationsv2')
sergio2_ds
#MODELS = list(set(sergio2_ds['validation']['model']))

README.md: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['bpb', 'grammar', 'ls_mean', 'ls_mean_w', 'ls_median', 'ls_median_w', 'ls_n', 'ls_perfect_pct', 'ls_perfect_pct_w', 'model', 'n_dedup_mean_per_story', 'n_dedup_total', 'n_err', 'n_stop', 'n_total', 'n_trunc_cot', 'n_trunc_post', 'parse_pct', 'parse_pct_total', 'parse_pct_total_w', 'parse_pct_w', 'pct_err', 'pct_err_w', 'pct_stop', 'pct_stop_w', 'pct_trunc_cot', 'pct_trunc_cot_w', 'pct_trunc_post', 'pct_trunc_post_w', 'split', 'status', 'ted_mean', 'ted_mean_w', 'ted_median', 'ted_median_w', 'ted_n', 'thinking'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['bpb', 'grammar', 'ls_mean', 'ls_mean_w', 'ls_median', 'ls_median_w', 'ls_n', 'ls_perfect_pct', 'ls_perfect_pct_w', 'model', 'n_dedup_mean_per_story', 'n_dedup_total', 'n_err', 'n_stop', 'n_total', 'n_trunc_cot', 'n_trunc_post', 'parse_pct', 'parse_pct_total', 'parse_pct_total_w', 'parse_pct_w', 'pct_err', 'pct_err_w', 'pct_stop', 'pct_stop_w', 'pct_trun

In [6]:
sergio2_ds['validation'][0]

{'bpb': 0.1253,
 'grammar': 'cfg',
 'ls_mean': 51.5217,
 'ls_mean_w': 54.4427,
 'ls_median': 45.0,
 'ls_median_w': 48.0,
 'ls_n': 69,
 'ls_perfect_pct': 0.0,
 'ls_perfect_pct_w': 0.0,
 'model': 'deepseek-r1-0528-qwen3-8b',
 'n_dedup_mean_per_story': 2.781,
 'n_dedup_total': 203,
 'n_err': 0,
 'n_stop': 69,
 'n_total': 73,
 'n_trunc_cot': 4,
 'n_trunc_post': 0,
 'parse_pct': 63.7681,
 'parse_pct_total': 60.274,
 'parse_pct_total_w': 64.532,
 'parse_pct_w': 68.2292,
 'pct_err': 0.0,
 'pct_err_w': 0.0,
 'pct_stop': 94.52,
 'pct_stop_w': 94.58,
 'pct_trunc_cot': 5.48,
 'pct_trunc_cot_w': 5.42,
 'pct_trunc_post': 0.0,
 'pct_trunc_post_w': 0.0,
 'split': 'validation',
 'status': 'ok',
 'ted_mean': 15.6136,
 'ted_mean_w': 15.0763,
 'ted_median': 12.5,
 'ted_median_w': 13.0,
 'ted_n': 44,
 'thinking': 'off'}

In [8]:
def get_cfg_generations(current_model, validation):
    """
    current_model = int ; Correspondiente a i in range(len(MODELS))
    validation = bool ; El booleano de siempre me lleva la verga.
    """
    if validation:
        dataset = sergio2_ds['validation']
        split = 'Validation'
    else:
        dataset = sergio2_ds['test']
        split = 'Test'

    LENGTH = len(dataset['model'])

    print("Filtrando al modelo: {}.".format(MODELS[current_model]))
    print("Split: {}.".format(split))
    trans_list = []
    for i in range(LENGTH):
        if (dataset['model'][i] == MODELS[current_model]) and dataset['setup'][i] == 'gen_cfg':
            premises = [text[2:] for text in dataset['answer_text'][i].split('\n')]
            if '' in premises:
                while '' in premises:
                    premises.remove('')
            trans_list.append(premises)

    df_dict = {'Translation': trans_list}
    dataframe = pd.DataFrame(data=df_dict)

    return dataframe


def evaluate_parsability_cfg(INDEX, validation):
    if validation:
        split = 'Validation'
    else:
        split = 'Test'

    dataset = get_cfg_generations(INDEX, validation)
    try:
        dataset = dataset.drop(columns = ["Unnamed: 0"])
    except:
        excepto = 0

    parse_total, nonparse_total, prover9ex_total, logicalex_total = 0, 0, 0, 0
    for i in range(len(dataset['Translation'])):
        parse, nonparse, prover9ex, logicalex  = check_arg_validity(dataset, i, validation)
        parse_total += parse
        nonparse_total += nonparse
        prover9ex_total += prover9ex
        logicalex_total += logicalex

    model_id = MODELS[INDEX]
    length = len(dataset['Translation'])

    print("-"*60)
    print('\t {}'.format(model_id))
    print('\t Split: {}'.format(split))
    print("-"*60)
    print("Valores parseables: {}".format(parse_total))
    print("Valores NO parseables: {}".format(nonparse_total))
    print("Porcentaje parseable: {}".format(round(parse_total/length, 3)))
    print("Arity Errors: {}".format(prover9ex_total))
    print("Syntax Errors: {}".format(logicalex_total))
    print('='*60)

In [9]:
for j in range(len(MODELS)):
    evaluate_parsability_cfg(j, True)
    evaluate_parsability_cfg(j, False)
    print('-'*60)

Filtrando al modelo: qwen3-4b-fp8.
Split: Validation.


LogicalExpressionException
Unexpected token: '⊕inactivedisinterested'.  Expected token ')'.
 all x (clubmember(x)&(performsintalentshow(x)⊕inactivedisinterested(x)))
                                              ^
LogicalExpressionException
Unexpected token: '⊕inactivedisinterested'.  Expected token ')'.
 all x (clubmember(x)&(performsintalentshow(x)⊕inactivedisinterested(x)))
                                              ^
LogicalExpressionException
Unexpected token: '⊕inactivedisinterested'.  Expected token ')'.
 all x (clubmember(x)&(performsintalentshow(x)⊕inactivedisinterested(x)))
                                              ^
LogicalExpressionException
Unexpected token: '⊕lunchathome'.  Expected token ')'.
allemployees(x)->(lunchinbuilding(x)⊕lunchathome(x))
                                    ^
LogicalExpressionException
Unexpected token: '⊕lunchathome'.  Expected token ')'.
allemployees(x)->(lunchinbuilding(x)⊕lunchathome(x))
                                    ^
LogicalExpre